# Set 5 오답노트

## 검토한 파일

- `set_01_06_answer/05_question.ipynb`
- `set_01_06_answer/05_answer.ipynb`
- `00_trying/01/05_question.ipynb`
- `00_trying/02/05_question.ipynb`

## 최종 답

- Q01: TENURE별 상관계수 중 최댓값 **0.95** — `TENURE=7`
- Q02: 군집별 일시불 구매액 평균 중 최댓값 **3946.19** — 최적 군집 수 `k=2`
- Q03: Measure B(RMSE) **1039.19**

현재 `00_trying/02/05_question.ipynb`의 세 최종 답은 모두 맞다. 이 노트는 풀이 과정에서 질문한 상관행렬·GroupBy apply·대각선 제거와 코드 주석에서 혼동한 부분을 중심으로 개선점을 정리한다.


## 공통 전처리 — `base` 만들기

원본 데이터는 1,000행, 18열이며 `MINIMUM_PAYMENTS`에만 결측치가 74개 있다. 문제에서 사용하는 `base`는 이 결측치를 해당 열의 평균으로 대체한 데이터다.

```python
base = df.copy()
base['MINIMUM_PAYMENTS'] = base['MINIMUM_PAYMENTS'].fillna(
    base['MINIMUM_PAYMENTS'].mean()
)
display(base.isna().sum().sum())  # 0
```

현재 풀이처럼 원본 `df`를 먼저 수정하고 `base = df.copy()`를 해도 결과는 같지만, 원본을 보존하려면 먼저 복사한 뒤 `base`에서 전처리하는 편이 안전하다.


## Q01 — TENURE별 Pearson 상관계수

### 문제의 분석 단위

문제는 전체 데이터의 상관관계 하나를 구하는 것이 아니다. `TENURE` 값이 6인 고객끼리, 7인 고객끼리, ..., 12인 고객끼리 나눈 다음 각 그룹 안에서 `BALANCE`와 `CREDIT_LIMIT`의 상관계수를 계산해야 한다.

```text
전체 DataFrame
    ↓ groupby('TENURE')
TENURE=6 그룹  → BALANCE ↔ CREDIT_LIMIT 상관계수
TENURE=7 그룹  → BALANCE ↔ CREDIT_LIMIT 상관계수
...
TENURE=12 그룹 → BALANCE ↔ CREDIT_LIMIT 상관계수
```

### `groupby().apply()`에서 객체가 변하는 과정

```python
corr_by_tenure = (
    df_q1.groupby('TENURE')[['BALANCE', 'CREDIT_LIMIT']]
    .apply(lambda group: group['BALANCE'].corr(group['CREDIT_LIMIT']))
)
```

객체 흐름:

```text
df_q1                                  DataFrame
df_q1.groupby('TENURE')                DataFrameGroupBy
...[['BALANCE', 'CREDIT_LIMIT']]       필요한 두 열만 쓰는 GroupBy
lambda의 group                         TENURE별 작은 DataFrame
group['BALANCE'].corr(...)             그룹마다 숫자 1개
apply 최종 결과                         TENURE 인덱스의 Series
```

`lambda group`의 `group`은 원본 전체가 아니라 현재 반복 중인 TENURE 값에 해당하는 작은 DataFrame이다. 상관계수는 두 열을 함께 사용하므로 열 하나를 단순 집계하는 `agg(sum/mean)`보다 각 그룹 DataFrame에 함수를 적용하는 `apply()`가 이해하기 쉽다.

### 실제 상관계수

```text
TENURE
6.0     0.868056
7.0     0.948405
8.0     0.820696
9.0     0.085474
10.0    0.291482
11.0    0.380360
12.0    0.460833
```

```python
best_tenure = corr_by_tenure.idxmax()
answer_q1 = round(corr_by_tenure.max(), 2)

display(best_tenure)  # 7.0
display(answer_q1)    # 0.95
```

문제는 가장 큰 **상관계수**를 요구하므로 `.abs()`를 적용하지 않는다. '상관계수 절댓값이 가장 큰 것'이라고 했다면 `.abs().max()`를 사용해야 한다.

### 현재 상관행렬 반환 방식은 가능한가?

현재 풀이의 함수는 그룹마다 숫자 하나가 아니라 2×2 상관행렬을 반환한다.

```python
def corr_(group):
    df_corr = group[['BALANCE', 'CREDIT_LIMIT']].corr()
    np.fill_diagonal(df_corr.values, 0)
    return df_corr
```

이 방식도 현재 데이터에서는 최종 `0.95`를 만들지만, 결과는 TENURE와 행 이름이 결합된 MultiIndex DataFrame이 되고 같은 상관계수가 대칭 위치에 두 번 들어간다. 최종 숫자 하나를 찾기 위해 `.max().max()`까지 호출해야 하므로 불필요하게 복잡하다.

또한 대각선을 0으로 바꾸면 상관계수가 음수일 때 문제가 생긴다. 예를 들어 실제 상관계수가 `-0.7`이면 행렬의 최댓값은 새로 넣은 대각선 0이 되어 잘못된 결과가 나온다. 대각선을 제외해야 한다면 0이 아니라 `np.nan`을 넣는다.

```python
np.fill_diagonal(df_corr.values, np.nan)
```

하지만 두 변수의 상관계수 위치를 이미 알고 있으므로 대각선을 수정할 필요 없이 직접 선택하는 것이 가장 정확하다.

```python
def corr_(group):
    df_corr = group[['BALANCE', 'CREDIT_LIMIT']].corr()
    return df_corr.loc['BALANCE', 'CREDIT_LIMIT']
```

`np.fill_diagonal()`은 전달받은 배열을 직접 수정하고 반환값은 `None`이다. 따라서 `dd = np.fill_diagonal(...)`에서 `dd`는 변경된 행렬이 아니라 `None`이다. DataFrame을 넘길 때는 `df_corr`가 아니라 값 배열 `df_corr.values`를 전달한다.

### 전체 상관행렬을 한 번 구하면 안 되는 이유

```python
df_q1[['BALANCE', 'TENURE', 'CREDIT_LIMIT']].corr()
```

이 코드는 TENURE별로 데이터를 나누지 않고 전체 1,000행에서 세 변수의 상관관계를 한 번만 계산한다. 실제 전체 `BALANCE ↔ CREDIT_LIMIT` 상관계수는 약 `0.467646`으로 문제의 답 `0.95`와 다르다. TENURE는 상관분석 대상 열이 아니라 그룹을 나누는 기준이다.

### 저장된 출력과 현재 코드가 다를 수 있음

현재 Q1 셀의 저장 출력에는 전체 상관행렬 결과 `0.467646`도 남아 있지만 현재 셀 소스에는 그 계산 코드가 없다. 이는 이전 코드를 실행한 뒤 일부 코드를 삭제하고 셀을 다시 실행하지 않아 생긴 오래된 출력이다. 제출 전에는 `Restart Kernel and Run All` 또는 최소한 변경한 셀을 다시 실행하여 코드와 출력이 일치하는지 확인한다.


## Q02 — 표준화, 최적 k, 원본 금액 평균

### 1. 고객 ID를 제외한 17개 변수 표준화

```python
X = base.drop(columns='CUST_ID').copy()
display(X.shape)  # (1000, 17)
display(X.dtypes) # 모두 숫자형인지 확인

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

`StandardScaler`는 각 열을 `(값-평균)/표준편차` 형태의 Z-score로 변환한다. 고객 번호는 크기 차이가 고객 특성을 의미하지 않는 식별자이므로 제외한다. 현재 17개 변수는 모두 숫자형이라 바로 표준화할 수 있지만, 다른 문제에서는 object 열이 없는지 먼저 확인해야 한다.

이번 문제는 train/test 분할 없이 전체 고객을 군집화하는 비지도학습이므로 전체 X에 `fit_transform()`을 사용하는 것이 맞다.

### 2. k=2~5의 실루엣 스코어 비교

```python
scores = {}
labels_by_k = {}

for k in range(2, 6):
    model = KMeans(n_clusters=k, random_state=1234, n_init=10)
    labels = model.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    labels_by_k[k] = labels

score_series = pd.Series(scores, name='silhouette')
best_k = score_series.idxmax()
```

실제 결과:

```text
k=2    0.307528  ← 최대
k=3    0.196361
k=4    0.207151
k=5    0.192741
```

실루엣 스코어는 클수록 같은 군집 안에서는 가깝고 다른 군집과는 잘 분리된 결과다. 따라서 `.idxmax()`로 최적 k `2`를 찾는다.

현재 주석의 `X['cluster'] = label[K] # dict을 바로 붙인다`에서 실제로 붙이는 것은 dict 전체가 아니다. `label`은 k를 키로 하고 라벨 배열을 값으로 저장한 dict이며, `label[K]`는 선택된 k에 대응하는 길이 1,000의 NumPy 배열이다.

```text
labels_by_k        → dict
labels_by_k[2]     → ndarray, shape (1000,)
```

### 3. 정규화하지 않은 금액으로 군집 평균 계산

군집을 만들 때는 표준화된 17개 변수를 사용하지만 최종 질문은 원래 화폐 단위의 `ONEOFF_PURCHASES` 평균을 요구한다. 따라서 라벨은 원본 X에 붙인다.

```python
X['cluster'] = labels_by_k[best_k]
oneoff_mean = X.groupby('cluster')['ONEOFF_PURCHASES'].mean()
answer_q2 = round(oneoff_mean.max(), 2)

display(oneoff_mean)
# cluster 0     340.230998
# cluster 1    3946.187525

display(answer_q2)  # 3946.19
```

군집 번호 0과 1은 크기나 우열을 나타내지 않는 임의의 라벨이다. 다른 초기화나 구현에서는 번호가 서로 바뀔 수 있으므로 특정 번호를 답으로 외우지 말고 `.max()`로 평균이 가장 큰 군집 값을 구한다.


## Q03 — 조건 분할, 의사결정나무, RMSE

### 1. 문제의 규칙대로 직접 분할

이 문제는 무작위 분할이 아니라 고객 ID의 4의 배수 여부로 분할한다. 따라서 `train_test_split()`을 사용하지 않는다.

```python
train = base.loc[base['CUST_ID'] % 4 != 0].copy()
test = base.loc[base['CUST_ID'] % 4 == 0].copy()

display(train.shape)  # (752, 18)
display(test.shape)   # (248, 18)
```

`CUST_ID.value_counts()`는 고객 ID가 중복되는지 확인하는 진단에는 사용할 수 있지만 문제 풀이에 필수적이지 않다.

### 2. X 16개와 1차원 y 만들기

```python
X_train = train.drop(columns=['CUST_ID', 'ONEOFF_PURCHASES']).copy()
X_test = test.drop(columns=['CUST_ID', 'ONEOFF_PURCHASES']).copy()
y_train = train['ONEOFF_PURCHASES'].copy()
y_test = test['ONEOFF_PURCHASES'].copy()

display(X_train.shape[1])  # 16
```

고객 ID는 식별자라서 제외하고, 예측 대상 `ONEOFF_PURCHASES`도 X에서 제외한다. 단일 종속변수 y는 이중 대괄호가 아닌 `df['column']` 형태의 Series로 만든다.

### 3. 의사결정나무는 표준화가 필요하지 않음

의사결정나무는 `특성값 <= 기준` 형태로 데이터를 나누므로 특성의 단위와 크기가 달라도 값의 순서가 유지되면 분할 결과가 변하지 않는다. Q2와 달리 Q3에서는 StandardScaler를 적용할 필요가 없다.

```python
model = DecisionTreeRegressor(random_state=1234)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

### 4. Measure B는 RMSE

문제의 Measure B는 오차를 제곱하고 평균을 낸 뒤 제곱근을 취하는 RMSE다.

```python
B = ((y_test - y_pred) ** 2).mean() ** 0.5
answer_q3 = round(B, 2)
display(answer_q3)  # 1039.19
```

sklearn 함수로 같은 계산을 할 수도 있다.

```python
from sklearn.metrics import mean_squared_error

B = mean_squared_error(y_test, y_pred) ** 0.5
```

RMSE는 원래 종속변수와 같은 단위를 가지며 작을수록 예측이 정확하다. `y_pred`는 X_test와 같은 행 순서로 생성되므로 현재 Series와 ndarray의 뺄셈도 정상 계산되지만, 명시적인 평가 함수가 코드 의도를 더 분명하게 보여준다.


## 핵심 암기

1. 그룹별 두 변수의 상관계수는 `groupby().apply()`에서 각 그룹 DataFrame의 두 Series에 `.corr()`를 적용한다.
2. 상관행렬에서 원하는 위치를 알면 대각선을 변경하지 말고 `.loc[row, column]`으로 직접 선택한다.
3. `np.fill_diagonal()`은 원본 배열을 수정하고 `None`을 반환하며, DataFrame에는 `.values`를 사용한다.
4. 대각선을 0으로 바꾸면 음의 상관계수에서 0이 잘못 최댓값이 될 수 있다.
5. TENURE별 분석과 전체 데이터의 상관행렬은 서로 다른 분석이다.
6. 군집분석에서는 식별자 CUST_ID를 제외하고 숫자형 17개 변수를 표준화한다.
7. 최적 k의 라벨 배열을 원본 금액 데이터에 붙여 원래 단위의 평균을 계산한다.
8. dict의 `labels_by_k[k]`는 dict 전체가 아니라 해당 k의 NumPy 라벨 배열이다.
9. 조건 기반 분할 문제에서는 임의의 `train_test_split()`으로 바꾸지 않는다.
10. 의사결정나무에는 일반적으로 특성 표준화가 필요하지 않다.
11. Measure B는 RMSE이며 작을수록 좋다.
12. 노트북 코드를 변경한 뒤에는 셀을 다시 실행해 저장 출력과 현재 코드를 일치시킨다.
